# So sánh 4 SOTA — Chỉ sử dụng kết quả đã chạy thực tế

Đọc bốn CSV, giữ `status=ok`. Nếu mô hình chưa chạy, không điền số giả định. **Không kết luận xếp hạng 4 SOTA khi chưa đủ 4 mô hình trên cùng seed/train_size.**

In [1]:
from pathlib import Path
import pandas as pd, numpy as np
ROOT=Path.cwd().resolve()
if not (ROOT/'results').exists(): ROOT=ROOT.parent
files=[ROOT/'results'/f for f in ['tabiclv2.csv','tabpfnv2.csv','tabdpt.csv','localpfn.csv']]
frames=[]
for p in files:
    if p.exists():
        f=pd.read_csv(p); frames.append(f)
        print(p.name,':',f['status'].value_counts().to_dict())
    else: print('CHƯA CÓ:',p.name)
all_results=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
if not all_results.empty: display(all_results)

tabiclv2.csv : {'ok': 2}
tabpfnv2.csv : {'ok': 2}
tabdpt.csv : {'ok': 2}
localpfn.csv : {'not_run': 2}


,model,dataset,seed,train_size,status,accuracy,macro_f1,roc_auc,log_loss,fit_seconds,predict_seconds,package_version,error
0,TabICLv2,breast_cancer_wisconsin,42,50,ok,0.973684,0.971863,0.996032,0.083052,25.869829,0.062738,2.2.0,NaN
1,TabICLv2,breast_cancer_wisconsin,42,100,ok,0.973684,0.971277,0.997685,0.064979,0.070166,0.067117,2.2.0,NaN
2,TabPFNv2,breast_cancer_wisconsin,42,50,ok,0.973684,0.971863,0.997024,0.078885,7.222088,0.148541,9.1.0,NaN
3,TabPFNv2,breast_cancer_wisconsin,42,100,ok,0.956140,0.952638,0.996693,0.080244,0.008273,0.167266,9.1.0,NaN
4,TabDPT,breast_cancer_wisconsin,42,50,ok,0.964912,0.961911,0.995701,0.101996,0.000722,0.153051,1.1.14,NaN
5,TabDPT,breast_cancer_wisconsin,42,100,ok,0.964912,0.961911,0.996362,0.091542,0.000753,0.198533,1.1.14,NaN
6,LoCalPFN,breast_cancer_wisconsin,42,50,not_run,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chưa clone official LoCalPFN repo vào vendor/L...
7,LoCalPFN,breast_cancer_wisconsin,42,100,not_run,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chưa clone official LoCalPFN repo vào vendor/L...


In [2]:
if all_results.empty:
    print('Chưa có kết quả chạy model thực tế. Hãy chạy các notebook 01–04 trước.')
else:
    ok=all_results[all_results['status'].eq('ok')].copy()
    if ok.empty:
        print('Chưa có dòng status=ok. Không vẽ bảng so sánh giả.')
    else:
        summary=ok.groupby(['model','train_size'],as_index=False).agg(
            n_seeds=('seed','nunique'),accuracy=('accuracy','mean'),
            macro_f1=('macro_f1','mean'),roc_auc=('roc_auc','mean'),
            log_loss=('log_loss','mean'),fit_seconds=('fit_seconds','mean'),
            predict_seconds=('predict_seconds','mean'))
        display(summary)
        summary.to_csv(ROOT/'results'/'summary_verified.csv',index=False)
        counts=ok.groupby(['seed','train_size'])['model'].nunique()
        print('Số cấu hình có đủ 4 mô hình:',int((counts==4).sum()))
        if not (counts==4).any():
            print('CHƯA THỂ KẾT LUẬN THỨ HẠNG 4 SOTA: thiếu kết quả cùng split.')

,model,train_size,n_seeds,accuracy,macro_f1,roc_auc,log_loss,fit_seconds,predict_seconds
0,TabDPT,50,1,0.964912,0.961911,0.995701,0.101996,0.000722,0.153051
1,TabDPT,100,1,0.964912,0.961911,0.996362,0.091542,0.000753,0.198533
2,TabICLv2,50,1,0.973684,0.971863,0.996032,0.083052,25.869829,0.062738
3,TabICLv2,100,1,0.973684,0.971277,0.997685,0.064979,0.070166,0.067117
4,TabPFNv2,50,1,0.973684,0.971863,0.997024,0.078885,7.222088,0.148541
5,TabPFNv2,100,1,0.956140,0.952638,0.996693,0.080244,0.008273,0.167266


Số cấu hình có đủ 4 mô hình: 0
CHƯA THỂ KẾT LUẬN THỨ HẠNG 4 SOTA: thiếu kết quả cùng split.


## Quy tắc diễn giải

- ROC-AUC chỉ có ý nghĩa khi test chứa cả hai lớp.
- So sánh accuracy/F1/AUC/log-loss trong **cùng seed và train_size**.
- Mức 50–100 mẫu là smoke test; mở rộng lên 5 seeds và 5–10 dataset trước khi viết kết luận nghiên cứu.
- Thời gian CPU không tương đương thời gian GPU công bố trong paper.
- Các model dùng `n_estimators=1` hoặc giảm context để chạy CPU: đây là **benchmark cấu hình CPU nhẹ**, không phải reproduction chính xác kết quả paper.